# 4 · Phonological normalisation of the IPA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/04_normalisation/04_phonological_normalisation.ipynb)

**Pipeline stage 4 of 6.** The book's semi-phonetic orthography carries
artefacts that are *spelling conventions*, not phonological facts. We apply
three rewrite rules and then tokenise each IPA word into phonemes against the
44-symbol Kölsch inventory, producing the phoneme-segmented references used to
train the recogniser.

**Rules**
- **(a) Degemination** — a doubled consonant only marks a short preceding vowel:
  `mʊttəʁ → mʊtəʁ`
- **(b) Double vowel → long vowel** — `vɛɛdən → vɛːdən`
- **(c) Silent *Dehnungs-h* → length** — `jəsaht → jəsaːt` (only when the *h* is
  pre-consonantal or word-final; an intervocalic *h* is kept, e.g. `dɔhɪn`).

## 1 · The Kölsch phoneme inventory (44 working symbols)

In [ ]:
PHONEME_MAPPING = {
    # long vowels
    'aː','ɛː','eː','iː','oː','uː','yː','øː',
    # diphthongs (German + Kölsch falling)
    'aɪ','aʊ','ɔɪ','ɛɪ','ɔʏ','ɐʊ','ɐɥ','ɐɪ',
    # affricates
    't͡s','p͡f','t͡ʃ',
    # short vowels
    'a','ɛ','ɪ','ɔ','ʊ','ʏ','œ','ə','ɐ','e','o','i','u','y','ø',
    # fricatives
    'ʃ','ʒ','ç','χ','x','f','v','s','z','h',
    # stops + glottal
    'p','b','t','d','k','ɡ','g','ʔ',
    # nasals
    'm','n','ŋ',
    # liquids / approximants
    'l','ʁ','j','r','w','ɥ',
}
_P = sorted(PHONEME_MAPPING, key=len, reverse=True)  # longest-match first
print(len(PHONEME_MAPPING), "phoneme symbols")

## 2 · The three normalisation rules

In [ ]:
import re

LONG = {'a':'aː','ɛ':'ɛː','e':'eː','i':'iː','o':'oː','u':'uː','y':'yː','ø':'øː','œ':'øː'}
CONS = set('pbtdkɡgʔmnŋlʁjrwɥʃʒçχxfvszh')

def rule_a_degeminate(s):
    """(a) collapse doubled consonants: mʊttəʁ -> mʊtəʁ"""
    out = []
    for i, ch in enumerate(s):
        if out and ch == out[-1] and ch in CONS:
            continue
        out.append(ch)
    return "".join(out)

def rule_b_double_vowel(s):
    """(b) double vowel -> long vowel: vɛɛdən -> vɛːdən"""
    for v, lv in LONG.items():
        s = s.replace(v + v, lv)
    return s

def rule_c_dehnungs_h(s):
    """(c) silent h after a vowel, pre-consonant or word-final -> length.
       Intervocalic h is kept (often a real /h/ at a morpheme boundary)."""
    VOW = "aɛɪɔʊʏœəɐeoiuyø"
    # vowel + h + (consonant | end)  ->  long vowel
    def repl(m):
        v = m.group(1)
        return LONG.get(v, v + 'ː')
    s = re.sub(rf"([{VOW}])h(?=[{''.join(CONS)}])", repl, s)
    s = re.sub(rf"([{VOW}])h$", repl, s)
    return s

def normalise_ipa(word):
    return rule_c_dehnungs_h(rule_b_double_vowel(rule_a_degeminate(word)))

for w in ["mʊttəʁ", "vɛɛdən", "jəsaht", "dɔhɪn"]:
    print(f"{w:10s} -> {normalise_ipa(w)}")

## 3 · Longest-match phoneme tokeniser

In [ ]:
def tokenize_word(word):
    """Split a normalised IPA word into phonemes (multi-char units first)."""
    out, i = [], 0
    while i < len(word):
        for p in _P:
            if word.startswith(p, i):
                out.append(p); i += len(p); break
        else:
            i += 1   # skip stress marks / unknown chars
    return out

def tokenize_ipa(text):
    """Whole IPA string -> 'p h o n | p h o n' (phonemes space-sep, words by |)."""
    words = [" ".join(tokenize_word(normalise_ipa(w))) for w in str(text).split()]
    return " | ".join(w for w in words if w)

print(tokenize_ipa("dat əsʊ jəsaht"))

## 4 · Apply to the corpus + zero-change safety pass

Run the normalisation over your IPA column, then re-verify with a safety pass:
re-tokenising an already-normalised string must produce **zero** further
changes (idempotence).

In [ ]:
# ---- Global: phonemise every utterance in the manifest ----
import os, pandas as pd
DATA="data"; SEG=f"{DATA}/segments"; MANIFEST=f"{SEG}/manifest.csv"; LEXICON=f"{DATA}/lexicon.tsv"

# Kölsch G2P via your pronunciation dictionary (word<TAB>ipa). Plug in your
# 29k-entry lexicon or a G2P model; ships empty in the 2-file example.
def load_lexicon(path=LEXICON):
    lex={}
    if os.path.exists(path):
        for line in open(path,encoding="utf-8"):
            if "\t" in line:
                w,ipa=line.rstrip("\n").split("\t",1); lex[w.lower()]=ipa
    return lex
LEX=load_lexicon()
def g2p(text):
    """utterance orthography -> word-form IPA via the lexicon (OOV words dropped)."""
    return " ".join(LEX[w] for w in str(text).lower().split() if w in LEX)

if os.path.exists(MANIFEST):
    man=pd.read_csv(MANIFEST)
    man["ipa_wordform"]=man["text"].map(g2p)               # orthography -> IPA words
    man["phonetic"]=man["ipa_wordform"].map(tokenize_ipa)  # -> phoneme tokens (rules applied)
    man.to_csv(MANIFEST,index=False)
    cov=(man["ipa_wordform"].str.len()>0).mean()
    print(f"phonemised {len(man)} utterances | lexicon coverage {cov*100:.0f}%")
    if cov<1: print("-> add data/lexicon.tsv (word<TAB>ipa) for full coverage")
    print(man[["id","text","ipa_wordform","phonetic"]].head().to_string())
else:
    demo=pd.DataFrame({"ipa":["dat əsʊ jəsaht","vɛɛl mʊttəʁ"]})
    demo["phonetic"]=demo["ipa"].map(tokenize_ipa)
    print("Run Notebook 3 first to build data/segments/manifest.csv.")
    print("Phonological-rules demo on sample IPA:\n", demo.to_string())

## Output

The `phonetic` column (`p h | p h` format) is the training label. Carry it into
the manifest produced in Notebook 3 (join on utterance id), then train in
Notebook 5. Keep the human-verified Kölsch pronunciation dictionary as the norm
and report deviations for expert judgement rather than auto-correcting.